# 03 Modeling and submissions (member B)

Versions are defined in `src/versions.py`, models in `src/models.py`.
Main-line versions (iter1-4) write `submissions/sub_<version>.csv`; breakdown versions run CV only. Both are logged in `reports/submission_log.csv`.
**After submitting on Kaggle, fill the Kaggle score into the `kaggle_score` column of `submission_log.csv`.**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURE_DIR, TARGET
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
from src.versions import VERSIONS
from src.pipeline import run_version, evaluate
for v, cfg in VERSIONS.items():
    print("CV    " if cfg["cv_only"] else "SUBMIT", v, cfg["desc"])

## 1. Run a version

In [ ]:
run_version("iter1_baseline")

## 2. Compare models (same features)

In [ ]:
base = VERSIONS["iter3_features"]
rows = []
for m in ["ridge", "lasso", "elasticnet", "rf", "gbr", "lgbm"]:
    mean, std = evaluate({**base, "model": m})
    rows.append({"model": m, "cv_rmse": mean, "cv_std": std})
    print(f"{m:<12} {mean:.5f} ± {std:.5f}")
models_df = pd.DataFrame(rows).sort_values("cv_rmse")
models_df

## 3. Tuning (e.g. Lasso alpha)

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from src.pipeline import prepare
from src.models import cross_validate

X, y, _, _ = prepare(base)
for alpha in [0.0001, 0.0003, 0.0005, 0.001, 0.003]:
    m = make_pipeline(RobustScaler(), Lasso(alpha=alpha, max_iter=50000))
    print(alpha, round(cross_validate(m, X, y, log_target=True)[0], 5))
# Once you find the best value, update MODELS["lasso"] in src/models.py

## 4. Score progress chart (for the report and slides)

In [ ]:
from src.config import SUBMISSION_LOG
full_log = pd.read_csv(SUBMISSION_LOG)
log = full_log[full_log.submitted].reset_index(drop=True)  # only iter1-4 submitted to Kaggle
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(log.version, log.cv_rmse, marker="o", label="CV")
if log.kaggle_score.notna().any():
    ax.plot(log.version, log.kaggle_score, marker="s", label="Kaggle")
ax.set_ylabel("RMSE (log)"); ax.set_title("Score by submission version"); ax.legend()
plt.tight_layout(); plt.savefig(FIGURE_DIR / "score_progress.png", dpi=150)
log

In [ ]:
# Biggest improvement (required report question)
log["improvement"] = -log.cv_rmse.diff()
log.sort_values("improvement", ascending=False)[["version", "description", "cv_rmse", "kaggle_score", "improvement"]]

## 5. Breakdown versions (CV only)

Separates the changes bundled inside each iter to show how much each one contributes.

In [ ]:
full_log[~full_log.submitted][["version", "description", "cv_rmse", "cv_std"]]